# Advanced Module 4 · Real-World Agents
Three complete agents doing real jobs — Travel Booking, Customer Support, Research Summarizer.
These become the building blocks for Module 5.

---
> **Setup:** Run the first two cells before anything else.

In [10]:
%pip install -q google-genai groq python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [11]:
import os, json, time, re, getpass, random
from dotenv import load_dotenv
from google import genai
from google.genai import types
from groq import Groq

load_dotenv()
gemini_api_key = os.environ.get('GEMINI_API_KEY') or getpass.getpass('Paste your Gemini API key: ')
groq_api_key   = (os.environ.get('Groq_key') or getpass.getpass('Paste your Groq API key: ')).strip()

gemini_client    = genai.Client(api_key=gemini_api_key)
groq_client      = Groq(api_key=groq_api_key)
GEMINI_MODEL     = 'gemini-3.1-flash-lite'
GROQ_FAST_MODEL  = 'llama-3.1-8b-instant'
DEFAULT_MAX_OUTPUT_TOKENS = 2048

def make_generation_config(**kwargs):
    """Build a GenerateContentConfig with sensible defaults.

    temperature=0.0  → deterministic; best for tool calls and routing decisions.
    temperature=0.7  → creative; best for final natural-language answers.
    """
    kwargs.setdefault('max_output_tokens', DEFAULT_MAX_OUTPUT_TOKENS)
    kwargs.setdefault('temperature', 0.7)
    return types.GenerateContentConfig(**kwargs)

def extract_text_from_response(response) -> str:
    """Pull the final answer text from a Gemini response, skipping reasoning thoughts."""
    if response.text:
        return response.text.strip()
    text_parts = []
    for candidate in (response.candidates or []):
        if candidate.content:
            for part in candidate.content.parts:
                if not getattr(part, 'thought', False) and part.text:
                    text_parts.append(part.text)
    return ''.join(text_parts).strip()

def call_with_retry(api_function, *args, max_retries=5, **kwargs):
    """Wrap an API call with automatic retry for rate-limit and server errors."""
    for attempt in range(max_retries):
        try:
            return api_function(*args, **kwargs)
        except Exception as error:
            error_message = str(error)
            if '429' in error_message or 'RESOURCE_EXHAUSTED' in error_message:
                retry_wait_match = re.search(r'retry[^0-9]*([0-9]+)s', error_message, re.I)
                wait_seconds = int(retry_wait_match.group(1)) + 5 if retry_wait_match else 35
                print(f'  ⏳ Rate-limited — waiting {wait_seconds}s')
                time.sleep(wait_seconds)
            elif '500' in error_message or 'INTERNAL' in error_message:
                time.sleep(10 * (attempt + 1))
            else:
                raise
    raise RuntimeError('Max retries exceeded')

original_generate_content = gemini_client.models.generate_content
gemini_client.models.generate_content = lambda *args, **kwargs: call_with_retry(original_generate_content, *args, **kwargs)

print('✅ Setup complete | Gemini:', GEMINI_MODEL, '| Groq fast:', GROQ_FAST_MODEL)

✅ Setup complete | Gemini: gemini-3.1-flash-lite | Groq fast: llama-3.1-8b-instant


In [12]:
try:
    test_response = gemini_client.models.generate_content(
        model=GEMINI_MODEL,
        contents='Reply with exactly the words: Hello LLM course',
        config=make_generation_config(temperature=0.0)
    )
    print('✅ API working:', extract_text_from_response(test_response))
except Exception as error:
    print('❌ Error:', error)

✅ API working: Hello LLM course


---
## Shared Agent Runner

All three agents below use the same ReAct loop from Module 2.  
Each agent only differs in its **system prompt**, **tool set**, and **tool implementations**.

In [13]:
def run_agent(
    task: str,
    system_prompt: str,
    tool_schema: types.Tool,
    tool_map: dict,
    max_steps: int = 10,
    label: str = 'Agent',
    verbose: bool = True
) -> str:
    """Generic ReAct agent runner — reused by all agents in this notebook.

    Builds a conversation from the system prompt and task, then loops:
      - If the LLM returns function_call parts → execute each tool, append results, repeat.
      - If the LLM returns text with no function_calls → that is the final answer. Stop.

    All three agents (Booking, Support, Research) share this loop.
    They differ only in the system_prompt, tool_schema, and tool_map they pass in.
    """
    conversation = [
        types.Content(role='user', parts=[types.Part(
            text=f'[SYSTEM]: {system_prompt}\n\n[USER TASK]: {task}'
        )])
    ]
    total_tokens = 0

    if verbose:
        print('\n' + '═' * 65)
        print(f'🤖 {label}')
        print(f'📋 Task: {task[:100]}...' if len(task) > 100 else f'📋 Task: {task}')
        print('═' * 65)

    for step_number in range(1, max_steps + 1):
        llm_response = gemini_client.models.generate_content(
            model=GEMINI_MODEL,
            contents=conversation,
            config=make_generation_config(tools=[tool_schema], temperature=0.2)
        )
        total_tokens += llm_response.usage_metadata.total_token_count

        all_function_calls = [
            part.function_call
            for part in llm_response.candidates[0].content.parts
            if hasattr(part, 'function_call') and part.function_call
        ]

        if not all_function_calls:
            final_answer = extract_text_from_response(llm_response)
            if verbose:
                print(f'\n[Step {step_number}] ✅ FINAL ANSWER:')
                print(final_answer)
                print(f'\n📊 Steps: {step_number} | Tokens: {total_tokens}')
            return final_answer

        tool_results = []
        for function_call in all_function_calls:
            if function_call.name not in tool_map:
                tool_results.append({'error': f'Unknown tool: {function_call.name}'})
                continue
            try:
                result = tool_map[function_call.name](**dict(function_call.args))
                tool_results.append(result)
                if verbose:
                    print(f'[Step {step_number}] 🔧 {function_call.name}({dict(function_call.args)})')
                    print(f'           → {json.dumps(result)[:120]}')
            except Exception as error:
                tool_results.append({'error': str(error)})
                if verbose:
                    print(f'[Step {step_number}] ❌ {function_call.name} ERROR: {error}')

        conversation.append(llm_response.candidates[0].content)  # preserves thought_signature
        conversation.append(types.Content(role='user', parts=[
            types.Part(function_response=types.FunctionResponse(
                name=function_call.name,
                response={'result': result}
            ))
            for function_call, result in zip(all_function_calls, tool_results)
        ]))

    return 'Max steps reached.'

print('✅ Agent runner ready')

✅ Agent runner ready


---
## Agent 1 — Travel Booking Agent

Handles end-to-end trip planning: flight search, hotel search, constraint reasoning, booking.

In [14]:
# Tool implementations
def search_flights(origin: str, destination: str, date: str) -> dict:
    options = [
        {'flight_id': 'SK201', 'airline': 'SkyWings',  'price_usd': 580, 'duration_h': 9,  'class': 'Economy'},
        {'flight_id': 'AE445', 'airline': 'AeroEast',  'price_usd': 640, 'duration_h': 11, 'class': 'Economy'},
        {'flight_id': 'SW900', 'airline': 'SwiftAir',  'price_usd': 520, 'duration_h': 10, 'class': 'Economy'},
        {'flight_id': 'LX101', 'airline': 'LuxAir',    'price_usd': 1200,'duration_h': 8,  'class': 'Business'},
    ]
    return {'route': f'{origin} → {destination}', 'date': date, 'options': options}

def search_hotels(city: str, checkin: str, checkout: str) -> dict:
    options = [
        {'hotel_id': 'H01', 'name': 'CityInn',        'price_per_night_usd': 85,  'rating': 3.8, 'stars': 3},
        {'hotel_id': 'H02', 'name': 'ParkView Grand', 'price_per_night_usd': 140, 'rating': 4.5, 'stars': 4},
        {'hotel_id': 'H03', 'name': 'BudgetStay',     'price_per_night_usd': 55,  'rating': 3.2, 'stars': 2},
        {'hotel_id': 'H04', 'name': 'Royal Suite',    'price_per_night_usd': 350, 'rating': 4.9, 'stars': 5},
    ]
    return {'city': city, 'checkin': checkin, 'checkout': checkout, 'options': options}

def book_itinerary(flight_id: str, hotel_id: str, passenger_name: str) -> dict:
    import random, string
    booking_ref = ''.join(random.choices(string.ascii_uppercase + string.digits, k=8))
    flight_prices = {'SK201': 580, 'AE445': 640, 'SW900': 520, 'LX101': 1200}
    hotel_prices  = {'H01': 85, 'H02': 140, 'H03': 55, 'H04': 350}
    return {
        'booking_ref':  booking_ref,
        'status':       'CONFIRMED',
        'passenger':    passenger_name,
        'flight_id':    flight_id,
        'hotel_id':     hotel_id,
        'flight_cost':  flight_prices.get(flight_id, 0),
        'hotel_cost':   hotel_prices.get(hotel_id, 0),
        'message':      f'Booking confirmed! Reference: {booking_ref}'
    }

def get_exchange_rate(from_currency: str, to_currency: str) -> dict:
    rates = {'USD': 1.0, 'EUR': 0.93, 'GBP': 0.79, 'INR': 83.5, 'AED': 3.67}
    rate  = rates.get(to_currency.upper(), 1.0) / rates.get(from_currency.upper(), 1.0)
    return {'from': from_currency, 'to': to_currency, 'rate': round(rate, 4)}

BOOKING_TOOLS = {
    'search_flights':   search_flights,
    'search_hotels':    search_hotels,
    'book_itinerary':   book_itinerary,
    'get_exchange_rate': get_exchange_rate,
}

booking_schema = types.Tool(function_declarations=[
    types.FunctionDeclaration(
        name='search_flights',
        description='Search for available flights. Returns multiple options with prices and duration.',
        parameters=types.Schema(type=types.Type.OBJECT, required=['origin','destination','date'],
            properties={
                'origin':      types.Schema(type=types.Type.STRING, description='Departure city'),
                'destination': types.Schema(type=types.Type.STRING, description='Arrival city'),
                'date':        types.Schema(type=types.Type.STRING, description='Travel date YYYY-MM-DD'),
            })
    ),
    types.FunctionDeclaration(
        name='search_hotels',
        description='Search for hotels in a city for given dates. Returns options with price and rating.',
        parameters=types.Schema(type=types.Type.OBJECT, required=['city','checkin','checkout'],
            properties={
                'city':     types.Schema(type=types.Type.STRING),
                'checkin':  types.Schema(type=types.Type.STRING, description='YYYY-MM-DD'),
                'checkout': types.Schema(type=types.Type.STRING, description='YYYY-MM-DD'),
            })
    ),
    types.FunctionDeclaration(
        name='book_itinerary',
        description='Confirm a booking with a specific flight and hotel. Returns a booking reference.',
        parameters=types.Schema(type=types.Type.OBJECT, required=['flight_id','hotel_id','passenger_name'],
            properties={
                'flight_id':      types.Schema(type=types.Type.STRING, description='Flight ID from search_flights'),
                'hotel_id':       types.Schema(type=types.Type.STRING, description='Hotel ID from search_hotels'),
                'passenger_name': types.Schema(type=types.Type.STRING),
            })
    ),
    types.FunctionDeclaration(
        name='get_exchange_rate',
        description='Get current exchange rate between two currencies.',
        parameters=types.Schema(type=types.Type.OBJECT, required=['from_currency','to_currency'],
            properties={
                'from_currency': types.Schema(type=types.Type.STRING, description='e.g. USD, EUR, INR'),
                'to_currency':   types.Schema(type=types.Type.STRING, description='e.g. USD, EUR, INR'),
            })
    ),
])

BOOKING_SYSTEM_PROMPT = """You are a smart travel booking assistant.
Help the user find and book the best trip based on their constraints (budget, dates, preferences).
Always search for flights and hotels before booking. Reason about the options and pick the best match.
When booking, confirm all details with the user's name."""

print('✅ Booking Agent tools defined')

✅ Booking Agent tools defined


In [15]:
run_agent(
    task=(
        "I'm Priya Sharma and I want to fly from Mumbai to Berlin on 2025-08-15 for 3 nights. "
        "My budget is $2000 total. I prefer a 4-star hotel. Book the best option for me."
    ),
    system_prompt=BOOKING_SYSTEM_PROMPT,
    tool_schema=booking_schema,
    tool_map=BOOKING_TOOLS,
    label='Travel Booking Agent'
)


═════════════════════════════════════════════════════════════════
🤖 Travel Booking Agent
📋 Task: I'm Priya Sharma and I want to fly from Mumbai to Berlin on 2025-08-15 for 3 nights. My budget is $2...
═════════════════════════════════════════════════════════════════
[Step 1] 🔧 search_flights({'destination': 'Berlin', 'origin': 'Mumbai', 'date': '2025-08-15'})
           → {"route": "Mumbai \u2192 Berlin", "date": "2025-08-15", "options": [{"flight_id": "SK201", "airline": "SkyWings", "price
[Step 2] 🔧 search_hotels({'checkin': '2025-08-15', 'checkout': '2025-08-18', 'city': 'Berlin'})
           → {"city": "Berlin", "checkin": "2025-08-15", "checkout": "2025-08-18", "options": [{"hotel_id": "H01", "name": "CityInn",
[Step 3] 🔧 book_itinerary({'flight_id': 'SK201', 'passenger_name': 'Priya Sharma', 'hotel_id': 'H02'})
           → {"booking_ref": "B8L8Y5S7", "status": "CONFIRMED", "passenger": "Priya Sharma", "flight_id": "SK201", "hotel_id": "H02",

[Step 4] ✅ FINAL ANSWER:
Hello Priy

'Hello Priya Sharma, I have successfully booked your trip to Berlin!\n\nHere are the details of your booking:\n*   **Flight:** SkyWings (Flight ID: SK201) from Mumbai to Berlin on 2025-08-15.\n*   **Hotel:** ParkView Grand (4-star hotel) for 3 nights (2025-08-15 to 2025-08-18).\n*   **Total Cost:** $1,000 ($580 for the flight + $420 for the hotel), which is well within your $2,000 budget.\n*   **Booking Reference:** B8L8Y5S7\n\nEnjoy your trip to Berlin!'

---
## Agent 2 — Customer Support Agent

Handles complaints, refunds, and escalations — with automatic escalation to a human when needed.

In [16]:
def lookup_order(order_id: str) -> dict:
    orders = {
        'ORD-1001': {'status': 'delivered', 'item': 'Laptop Pro 15"',  'price': 1299, 'delivery_date': '2025-07-10'},
        'ORD-1002': {'status': 'in_transit', 'item': 'Wireless Headphones', 'price': 249, 'eta': '2025-07-20'},
        'ORD-1003': {'status': 'cancelled',  'item': 'Smart Watch',     'price': 399, 'cancel_reason': 'out_of_stock'},
    }
    return orders.get(order_id, {'error': f'Order {order_id} not found'})

def check_refund_policy(reason: str) -> dict:
    policies = {
        'defective':  {'eligible': True,  'window_days': 30, 'method': 'full refund or replacement'},
        'not_needed': {'eligible': True,  'window_days': 14, 'method': 'refund minus 10% restocking fee'},
        'late':       {'eligible': True,  'window_days': 7,  'method': 'full refund'},
        'fraud':      {'eligible': True,  'window_days': 60, 'method': 'immediate full refund'},
        'other':      {'eligible': False, 'window_days': 0,  'method': 'escalate to human agent'},
    }
    return policies.get(reason.lower(), policies['other'])

def raise_ticket(order_id: str, issue_type: str, description: str) -> dict:
    import random
    ticket_id = f'TKT-{random.randint(10000, 99999)}'
    return {'ticket_id': ticket_id, 'status': 'open', 'priority': 'normal', 'sla_hours': 24}

def escalate_to_human(reason: str, order_id: str) -> dict:
    return {
        'escalated':    True,
        'queue':        'tier-2-support',
        'wait_time_min': 15,
        'message':      f'Your case ({order_id}) has been escalated. A human agent will contact you shortly. Reason: {reason}'
    }

SUPPORT_TOOLS = {
    'lookup_order':     lookup_order,
    'check_refund_policy': check_refund_policy,
    'raise_ticket':     raise_ticket,
    'escalate_to_human': escalate_to_human,
}

support_schema = types.Tool(function_declarations=[
    types.FunctionDeclaration(
        name='lookup_order', description='Look up order details by order ID.',
        parameters=types.Schema(type=types.Type.OBJECT, required=['order_id'],
            properties={'order_id': types.Schema(type=types.Type.STRING)})
    ),
    types.FunctionDeclaration(
        name='check_refund_policy',
        description='Check refund policy for a given return reason.',
        parameters=types.Schema(type=types.Type.OBJECT, required=['reason'],
            properties={'reason': types.Schema(type=types.Type.STRING, description='defective / not_needed / late / fraud / other')})
    ),
    types.FunctionDeclaration(
        name='raise_ticket',
        description='Create a support ticket for an order issue.',
        parameters=types.Schema(type=types.Type.OBJECT, required=['order_id','issue_type','description'],
            properties={
                'order_id':    types.Schema(type=types.Type.STRING),
                'issue_type':  types.Schema(type=types.Type.STRING),
                'description': types.Schema(type=types.Type.STRING),
            })
    ),
    types.FunctionDeclaration(
        name='escalate_to_human',
        description='Escalate to a human agent when the issue is too complex or sensitive.',
        parameters=types.Schema(type=types.Type.OBJECT, required=['reason','order_id'],
            properties={
                'reason':   types.Schema(type=types.Type.STRING, description='Why escalation is needed'),
                'order_id': types.Schema(type=types.Type.STRING),
            })
    ),
])

SUPPORT_SYSTEM_PROMPT = """You are a professional customer support agent.
Be empathetic and solution-focused. Always look up the order first before making decisions.
If the issue is complex, uncertain, or the customer is very upset, escalate to a human agent.
Always raise a ticket for any issue so it's tracked."""

print('✅ Customer Support Agent tools defined')

✅ Customer Support Agent tools defined


In [17]:
run_agent(
    task=(
        "I received my laptop from order ORD-1001 and it's completely broken — "
        "the screen doesn't turn on. I paid $1299 for this and I'm furious. "
        "I want a full refund immediately!"
    ),
    system_prompt=SUPPORT_SYSTEM_PROMPT,
    tool_schema=support_schema,
    tool_map=SUPPORT_TOOLS,
    label='Customer Support Agent'
)


═════════════════════════════════════════════════════════════════
🤖 Customer Support Agent
📋 Task: I received my laptop from order ORD-1001 and it's completely broken — the screen doesn't turn on. I ...
═════════════════════════════════════════════════════════════════
[Step 1] 🔧 lookup_order({'order_id': 'ORD-1001'})
           → {"status": "delivered", "item": "Laptop Pro 15\"", "price": 1299, "delivery_date": "2025-07-10"}
[Step 2] 🔧 check_refund_policy({'reason': 'defective'})
           → {"eligible": true, "window_days": 30, "method": "full refund or replacement"}
[Step 3] 🔧 raise_ticket({'order_id': 'ORD-1001', 'issue_type': 'defective_item', 'description': 'Customer received a broken laptop (screen does not turn on). Customer is very upset and demanding a full refund.'})
           → {"ticket_id": "TKT-84270", "status": "open", "priority": "normal", "sla_hours": 24}
[Step 4] 🔧 escalate_to_human({'reason': 'Customer is extremely upset and demanding an immediate refund for a high

"I am incredibly sorry to hear that your new laptop arrived in this condition. I completely understand your frustration—spending $1299 on a device that doesn't turn on is unacceptable, and I sincerely apologize for this experience.\n\nI have looked into your order (ORD-1001) and have taken the following steps to resolve this for you:\n\n1.  **Ticket Created:** I have opened a support ticket (**TKT-84270**) to document this issue and ensure it is tracked.\n2.  **Refund Eligibility:** I have verified our policy, and because the item is defective, you are eligible for a full refund or a replacement.\n3.  **Escalation:** Given how upset you are and the high value of the item, I have escalated your case directly to our senior support team. They are best equipped to handle this immediately and will ensure your refund is processed as quickly as possible.\n\nA human agent will contact you within the next 15 minutes to finalize the next steps. Thank you for your patience while we make this righ

---
## Agent 3 — Research Summarizer Agent

Fetches articles, extracts key claims, cross-checks for consistency, and writes a structured report.

In [18]:
# Stub article database
ARTICLE_DB = {
    'https://techblog.example.com/ai-agents-2025': {
        'title': 'The Rise of AI Agents in 2025',
        'text': (
            'AI agents are now handling 40% of customer support queries at Fortune 500 companies. '
            'Multi-agent systems have shown 3x productivity gains in software development tasks. '
            'However, agent hallucination rates remain at 8-12% without proper grounding.'
        )
    },
    'https://research.example.com/llm-productivity': {
        'title': 'LLM Impact on Developer Productivity',
        'text': (
            'Studies show developers using AI coding assistants complete tasks 55% faster. '
            'Code review time has dropped by 30% with LLM-assisted analysis. '
            'AI agents account for 25% of new startup codebases according to Y Combinator.'
        )
    },
    'https://safety.example.com/agent-risks': {
        'title': 'Safety Risks in Agentic AI Systems',
        'text': (
            'Prompt injection attacks have been demonstrated on 15 production agent systems. '
            'Agents with unrestricted tool access caused $2.3M in accidental costs in 2024. '
            'Researchers recommend strict tool allowlists and human-in-the-loop for irreversible actions.'
        )
    }
}

def fetch_article(url: str) -> dict:
    article = ARTICLE_DB.get(url)
    if not article:
        return {'error': f'Article not found: {url}'}
    return {'url': url, 'title': article['title'], 'content': article['text'], 'word_count': len(article['text'].split())}

def extract_key_claims(text: str) -> dict:
    """Use Groq (fast) to extract key claims from text."""
    resp = groq_client.chat.completions.create(
        model=GROQ_FAST,
        messages=[{
            'role': 'user',
            'content': f'Extract 2-3 key factual claims from this text as a JSON array of strings:\n\n{text}'
        }],
        max_tokens=256,
        temperature=0.0
    )
    content = resp.choices[0].message.content
    # Try to parse JSON, fall back to lines
    try:
        match = re.search(r'\[.*?\]', content, re.DOTALL)
        claims = json.loads(match.group(0)) if match else [content]
    except Exception:
        claims = [line.strip() for line in content.splitlines() if line.strip()]
    return {'claims': claims, 'count': len(claims)}

def fact_check(claim: str) -> dict:
    """Use Groq to assess claim credibility."""
    resp = groq_client.chat.completions.create(
        model=GROQ_FAST,
        messages=[{
            'role': 'user',
            'content': (
                f'Rate this claim as LIKELY_TRUE, UNCERTAIN, or LIKELY_FALSE, with a 1-sentence reason.\n\n'
                f'Claim: {claim}\n\nFormat: {{"verdict": "...", "reason": "..."}}'
            )
        }],
        max_tokens=128,
        temperature=0.0
    )
    content = resp.choices[0].message.content
    try:
        match = re.search(r'\{.*?\}', content, re.DOTALL)
        result = json.loads(match.group(0)) if match else {'verdict': 'UNCERTAIN', 'reason': content}
    except Exception:
        result = {'verdict': 'UNCERTAIN', 'reason': content[:100]}
    return {**result, 'claim': claim}

def list_available_articles(topic: str) -> dict:
    """Return available article URLs for a topic."""
    return {
        'topic': topic,
        'available_urls': list(ARTICLE_DB.keys()),
        'count': len(ARTICLE_DB)
    }

RESEARCH_TOOLS = {
    'list_available_articles': list_available_articles,
    'fetch_article':           fetch_article,
    'extract_key_claims':      extract_key_claims,
    'fact_check':              fact_check,
}

research_schema = types.Tool(function_declarations=[
    types.FunctionDeclaration(
        name='list_available_articles',
        description='List available article URLs for a given research topic.',
        parameters=types.Schema(type=types.Type.OBJECT, required=['topic'],
            properties={'topic': types.Schema(type=types.Type.STRING)})
    ),
    types.FunctionDeclaration(
        name='fetch_article',
        description='Fetch the full text of an article given its URL.',
        parameters=types.Schema(type=types.Type.OBJECT, required=['url'],
            properties={'url': types.Schema(type=types.Type.STRING)})
    ),
    types.FunctionDeclaration(
        name='extract_key_claims',
        description='Extract 2-3 key factual claims from a block of text.',
        parameters=types.Schema(type=types.Type.OBJECT, required=['text'],
            properties={'text': types.Schema(type=types.Type.STRING)})
    ),
    types.FunctionDeclaration(
        name='fact_check',
        description='Assess the credibility of a single factual claim.',
        parameters=types.Schema(type=types.Type.OBJECT, required=['claim'],
            properties={'claim': types.Schema(type=types.Type.STRING)})
    ),
])

RESEARCH_SYSTEM_PROMPT = """You are a rigorous research analyst.
For any research task: list available articles, fetch them, extract key claims, and fact-check each one.
Synthesize a structured report that includes verified claims, uncertain claims, and a summary."""

print('✅ Research Summarizer Agent tools defined')

✅ Research Summarizer Agent tools defined


In [19]:
run_agent(
    task='Research the current state of AI agents in 2025. Fetch available articles, extract claims, fact-check them, and write a structured summary.',
    system_prompt=RESEARCH_SYSTEM_PROMPT,
    tool_schema=research_schema,
    tool_map=RESEARCH_TOOLS,
    label='Research Summarizer Agent',
    max_steps=12
)


═════════════════════════════════════════════════════════════════
🤖 Research Summarizer Agent
📋 Task: Research the current state of AI agents in 2025. Fetch available articles, extract claims, fact-chec...
═════════════════════════════════════════════════════════════════
[Step 1] 🔧 list_available_articles({'topic': 'current state of AI agents 2025'})
           → {"topic": "current state of AI agents 2025", "available_urls": ["https://techblog.example.com/ai-agents-2025", "https://
[Step 2] 🔧 fetch_article({'url': 'https://techblog.example.com/ai-agents-2025'})
           → {"url": "https://techblog.example.com/ai-agents-2025", "title": "The Rise of AI Agents in 2025", "content": "AI agents a
[Step 3] 🔧 fetch_article({'url': 'https://research.example.com/llm-productivity'})
           → {"url": "https://research.example.com/llm-productivity", "title": "LLM Impact on Developer Productivity", "content": "St
[Step 4] 🔧 fetch_article({'url': 'https://safety.example.com/agent-risks'})
    

'# Research Report: The State of AI Agents in 2025\n\nThis report synthesizes findings from current industry literature regarding the adoption, productivity impact, and safety risks of AI agents in 2025.\n\n## Verified and Likely Claims\n*   **Productivity Gains in Development:** Research indicates that developers utilizing AI coding assistants complete tasks significantly faster (up to 55% faster in some studies). This is supported by consistent industry reports on the integration of LLMs into software development workflows.\n\n## Uncertain Claims\n*   **Customer Support Adoption:** The claim that AI agents handle 40% of customer support queries at Fortune 500 companies lacks verifiable industry-wide data. While adoption is high, this specific figure is likely an estimate or specific to a subset of companies rather than a universal standard.\n*   **Financial Impact of Agent Errors:** The claim that agents with unrestricted tool access caused $2.3M in accidental costs in 2024 is curren

---
## Agent 4 — Human-in-the-Loop (HITL) Booking Agent

Some tool calls are **irreversible** — once a booking is confirmed or a record is deleted, there is no undo.  
The **Human-in-the-Loop** pattern inserts a human checkpoint before any tool marked as *gated*.

### How it works

```
Agent reasons → wants to call book_itinerary
        ↓
 ⚠️  PAUSE — show human the exact args
        ↓
  Human types y / n
        ↓
  y → tool executes normally
  n → agent receives {"status": "rejected"} and must reason about what to do next
```

This is the **same ReAct loop** from Module 2.  
The only addition is a single approval check inside the tool-dispatch block.

| Scenario | What the agent receives | Agent behaviour |
|---|---|---|
| Human approves | Real tool result | Continues normally |
| Human rejects | `{"status": "rejected", "message": "..."}` | Re-plans: suggest alternatives or stop |

In [20]:
# Tools that require explicit human approval before execution.
# Add any irreversible tool here: bookings, payments, deletions, emails.
GATED_TOOL_NAMES = {'book_itinerary'}

def run_agent_with_human_approval(
    task: str,
    system_prompt: str,
    tool_schema: types.Tool,
    tool_map: dict,
    gated_tools: set,
    max_steps: int = 10,
    label: str = 'HITL Agent',
    verbose: bool = True
) -> str:
    """ReAct agent with a human-in-the-loop approval gate for irreversible tools.

    Identical to run_agent() except that before executing any tool whose name is
    in `gated_tools`, the agent pauses and asks the human to approve (y) or reject (n).

    On approval  → the tool runs normally and the result is returned to the LLM.
    On rejection → the LLM receives {'status': 'rejected', 'message': '...'} as the
                   tool result and must reason about what to do next (suggest alternatives,
                   ask for clarification, or stop). The agent never crashes on a rejection.

    Why feed rejection back as a tool result?
    Because it keeps the agent inside its normal ReAct loop. The LLM already knows how
    to handle unexpected tool results — it reasons about them and adapts. We don't need
    a special code path.
    """
    conversation = [
        types.Content(role='user', parts=[types.Part(
            text=f'[SYSTEM]: {system_prompt}\n\n[USER TASK]: {task}'
        )])
    ]
    total_tokens = 0

    if verbose:
        print('\n' + '═' * 65)
        print(f'🤖 {label}  (HITL mode — gated tools: {gated_tools})')
        print(f'📋 Task: {task[:100]}...' if len(task) > 100 else f'📋 Task: {task}')
        print('═' * 65)

    for step_number in range(1, max_steps + 1):
        llm_response = gemini_client.models.generate_content(
            model=GEMINI_MODEL,
            contents=conversation,
            config=make_generation_config(tools=[tool_schema], temperature=0.2)
        )
        total_tokens += llm_response.usage_metadata.total_token_count

        all_function_calls = [
            part.function_call
            for part in llm_response.candidates[0].content.parts
            if hasattr(part, 'function_call') and part.function_call
        ]

        if not all_function_calls:
            final_answer = extract_text_from_response(llm_response)
            if verbose:
                print(f'\n[Step {step_number}] ✅ FINAL ANSWER:')
                print(final_answer)
                print(f'\n📊 Steps: {step_number} | Tokens: {total_tokens}')
            return final_answer

        tool_results = []
        for function_call in all_function_calls:
            if function_call.name not in tool_map:
                tool_results.append({'error': f'Unknown tool: {function_call.name}'})
                continue

            call_args = dict(function_call.args)

            # ── Human-in-the-loop approval gate ──────────────────────────
            if function_call.name in gated_tools:
                print(f'\n[Step {step_number}] ⚠️  APPROVAL REQUIRED — {function_call.name}')
                print(f'           Args: {json.dumps(call_args, indent=12)}')
                human_input = input('           Approve? [y/n]: ').strip().lower()
                human_approved = human_input == 'y'

                if not human_approved:
                    rejection_result = {
                        'status':  'rejected',
                        'message': (
                            f'Human rejected the call to {function_call.name}. '
                            'Do NOT retry this action. Suggest alternatives or stop.'
                        )
                    }
                    tool_results.append(rejection_result)
                    if verbose:
                        print(f'           ✋ Rejected — sending back: {rejection_result}')
                    continue

                if verbose:
                    print(f'           ✅ Approved — executing ...')
            # ─────────────────────────────────────────────────────────────

            try:
                result = tool_map[function_call.name](**call_args)
                tool_results.append(result)
                if verbose:
                    print(f'[Step {step_number}] 🔧 {function_call.name}({call_args})')
                    print(f'           → {json.dumps(result)[:120]}')
            except Exception as error:
                tool_results.append({'error': str(error)})
                if verbose:
                    print(f'[Step {step_number}] ❌ {function_call.name} ERROR: {error}')

        conversation.append(llm_response.candidates[0].content)  # preserves thought_signature
        conversation.append(types.Content(role='user', parts=[
            types.Part(function_response=types.FunctionResponse(
                name=function_call.name,
                response={'result': result}
            ))
            for function_call, result in zip(all_function_calls, tool_results)
        ]))

    return 'Max steps reached.'

print('✅ HITL agent runner ready')

✅ HITL agent runner ready


### Demo — Run the HITL Booking Agent

Two scenarios below. When prompted **`Approve? [y/n]`**, type:
- **`y`** → booking executes and you get a confirmation reference
- **`n`** → agent receives a rejection and must re-plan (suggest alternatives / stop)

In [21]:
# ── Scenario A: Human APPROVES the booking ────────────────────────────────
# When prompted "Approve? [y/n]" → type  y  and press Enter
# Expected: booking confirms and you get a reference number

run_agent_with_human_approval(
    task=(
        "I'm Arjun Mehta. Fly me from Delhi to Paris on 2025-09-10 for 4 nights. "
        "Budget is $2500 total. Prefer economy class and a 4-star hotel. Book the best option."
    ),
    system_prompt=BOOKING_SYSTEM_PROMPT,
    tool_schema=booking_schema,
    tool_map=BOOKING_TOOLS,
    gated_tools=GATED_TOOL_NAMES,
    label='HITL Booking Agent — Approve demo'
)


═════════════════════════════════════════════════════════════════
🤖 HITL Booking Agent — Approve demo  (HITL mode — gated tools: {'book_itinerary'})
📋 Task: I'm Arjun Mehta. Fly me from Delhi to Paris on 2025-09-10 for 4 nights. Budget is $2500 total. Prefe...
═════════════════════════════════════════════════════════════════
[Step 1] 🔧 search_flights({'destination': 'Paris', 'date': '2025-09-10', 'origin': 'Delhi'})
           → {"route": "Delhi \u2192 Paris", "date": "2025-09-10", "options": [{"flight_id": "SK201", "airline": "SkyWings", "price_u
[Step 2] 🔧 search_hotels({'checkout': '2025-09-14', 'city': 'Paris', 'checkin': '2025-09-10'})
           → {"city": "Paris", "checkin": "2025-09-10", "checkout": "2025-09-14", "options": [{"hotel_id": "H01", "name": "CityInn", 

[Step 3] ⚠️  APPROVAL REQUIRED — book_itinerary
           Args: {
            "passenger_name": "Arjun Mehta",
            "hotel_id": "H02",
            "flight_id": "SW900"
}
           ✅ Approved — executing ...

'Hello Arjun Mehta, I have successfully booked your trip to Paris.\n\n**Booking Details:**\n*   **Passenger:** Arjun Mehta\n*   **Flight:** SwiftAir (Economy), Delhi to Paris on 2025-09-10 ($520)\n*   **Hotel:** ParkView Grand (4-star), 4 nights from 2025-09-10 to 2025-09-14 ($560 total)\n*   **Total Cost:** $1,080 (well within your $2,500 budget)\n*   **Booking Reference:** ZHA6DY25\n\nHave a wonderful trip!'

In [24]:
# ── Scenario B: Human REJECTS the booking ─────────────────────────────────
# When prompted "Approve? [y/n]" → type  n  and press Enter
# Expected: agent gets a rejection result and suggests alternatives instead of booking

run_agent_with_human_approval(
    task=(
        "I'm Meera Nair. Book me a flight from Chennai to London on 2025-10-05 for 2 nights. "
        "Any hotel is fine. Just get the cheapest option and confirm it."
    ),
    system_prompt=BOOKING_SYSTEM_PROMPT,
    tool_schema=booking_schema,
    tool_map=BOOKING_TOOLS,
    gated_tools=GATED_TOOL_NAMES,
    label='HITL Booking Agent — Reject demo'
)


═════════════════════════════════════════════════════════════════
🤖 HITL Booking Agent — Reject demo  (HITL mode — gated tools: {'book_itinerary'})
📋 Task: I'm Meera Nair. Book me a flight from Chennai to London on 2025-10-05 for 2 nights. Any hotel is fin...
═════════════════════════════════════════════════════════════════
[Step 1] 🔧 search_flights({'destination': 'London', 'origin': 'Chennai', 'date': '2025-10-05'})
           → {"route": "Chennai \u2192 London", "date": "2025-10-05", "options": [{"flight_id": "SK201", "airline": "SkyWings", "pric
[Step 1] 🔧 search_hotels({'city': 'London', 'checkout': '2025-10-07', 'checkin': '2025-10-05'})
           → {"city": "London", "checkin": "2025-10-05", "checkout": "2025-10-07", "options": [{"hotel_id": "H01", "name": "CityInn",

[Step 2] ⚠️  APPROVAL REQUIRED — book_itinerary
           Args: {
            "flight_id": "SW900",
            "hotel_id": "H03",
            "passenger_name": "Meera Nair"
}
           ✋ Rejected — sending bac

'I have found the cheapest options for your trip from Chennai to London (2025-10-05 to 2025-10-07):\n\n*   **Flight:** SwiftAir (SW900) at $520.\n*   **Hotel:** BudgetStay (H03) at $55 per night.\n\nHowever, I encountered an issue when attempting to finalize the booking. Please let me know if you would like me to proceed with these specific options, or if you would prefer to look at different flights or hotels.'

In [23]:
# ✏️  Add a new tool to one of the agents above.
#     Ideas for Booking Agent: check_baggage_policy(airline), get_airport_info(city)
#     Ideas for Support Agent: get_loyalty_points(customer_id), send_voucher(email, amount)
#     Ideas for Research Agent: summarize_with_bullets(text), compare_sources(url1, url2)

def my_new_tool(input_param: str) -> dict:
    # TODO: implement your tool
    return {'result': f'Processed: {input_param}'}

print('Define your tool and add it to one of the TOOL_MAP dicts above, then re-run the agent.')

Define your tool and add it to one of the TOOL_MAP dicts above, then re-run the agent.


---
## Key Takeaways

| Agent | What makes it real-world? |
|---|---|
| Travel Booking | Constraint reasoning (budget, rating), multi-hop (search → select → book) |
| Customer Support | Empathy + escalation trigger, policy lookup, ticket tracking |
| Research Summarizer | Uses Groq for fast sub-tasks, fact-checks every claim before reporting |
| **HITL Booking** | **Approval gate before irreversible tools; rejection fed back so agent re-plans** |

| Pattern | Applies to |
|---|---|
| Same `run_agent()` loop for all 3 | Architecture is reusable; agents differ only in system prompt + tools |
| Groq for fast sub-tasks | Use fast small models for classification/extraction; Gemini for reasoning |
| These are worker agents | Module 5 will orchestrate them together under a single orchestrator |
| `GATED_TOOLS` set | Any tool whose side-effects can't be undone belongs here — bookings, deletes, payments |
| Rejection as a tool result | Feeding `{"status": "rejected"}` back to the LLM lets it reason about alternatives without a crash |